# GPU-Accelerated Coin Counting

**Task:** Object counting (coins) from images.

**Pipeline in this notebook:**
1. Preprocessing
2. Classical computer-vision detector (feature extraction + Hough Circle Transform) — a fast, training-free baseline
3. Auto-labelling: turn the classical detector's output into YOLO-format bounding-box labels (since the raw dataset has no annotations)
4. Data augmentation to grow the small (<100 image) dataset
5. GPU-accelerated deep-learning detector: fine-tune YOLOv8n (Ultralytics/PyTorch, CUDA) on the augmented, auto-labelled set
6. Evaluation: classical vs. deep-learning counts vs. your own manual ground truth

**Running locally in VS Code:** see `README.md` for environment setup (venv + `pip install -r requirements.txt`).
If you have an NVIDIA GPU with CUDA drivers installed, training will use it automatically; otherwise it falls back to CPU (slower, but the notebook still runs end to end).


In [1]:
import cv2
import numpy as np
import os, glob, shutil, random, json
import matplotlib.pyplot as plt
import torch

print("OpenCV:", cv2.__version__)
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No CUDA GPU detected - training will run on CPU (Section 7 will be slow).")

random.seed(42)
np.random.seed(42)


OpenCV: 5.0.0
Torch: 2.11.0+cu128 | CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## 1. Load the dataset

This dataset is organised as one subfolder per coin count, e.g.:

```
data/coin_images/
    1/    (10 images, each containing exactly 1 coin)
    2/    (10 images, each containing exactly 2 coins)
    ...
    29/   (10 images, each containing exactly 29 coins)
```

The folder name **is** the ground-truth coin count for every image inside it —
so we get exact, free labels for all 300 images (no manual counting needed).
We keep each image's path relative to `DATA_DIR` (e.g. `"7/IMG003.jpg"`) as its
key everywhere in this notebook, rather than just the filename, since filenames
repeat across folders (`"1/a.jpg"` and `"2/a.jpg"` are different images).


In [2]:
# EDIT THIS to the local folder containing your coin_images/<count>/*.jpg structure
DATA_DIR = r"./data/coin_images"

image_records = []  # list of (absolute_path, true_count)
for sub in sorted(os.listdir(DATA_DIR), key=lambda s: (len(s), s)):
    sub_path = os.path.join(DATA_DIR, sub)
    if not os.path.isdir(sub_path):
        continue
    try:
        true_count = int(sub)
    except ValueError:
        print(f"Skipping non-numeric folder: {sub}")
        continue
    files = (glob.glob(os.path.join(sub_path, "*.jpg")) +
             glob.glob(os.path.join(sub_path, "*.jpeg")) +
             glob.glob(os.path.join(sub_path, "*.png")))
    for p in sorted(files):
        image_records.append((p, true_count))

image_paths = [p for p, _ in image_records]
# ground truth is auto-derived from folder names - keyed by path relative to DATA_DIR
ground_truth = {os.path.relpath(p, DATA_DIR): c for p, c in image_records}

print(f"Found {len(image_records)} images across {len(set(ground_truth.values()))} distinct coin counts")
assert len(image_records) > 0, "No images found - check DATA_DIR and folder naming"

# quick look at a few
fig, axes = plt.subplots(1, min(4, len(image_paths)), figsize=(16, 4))
for ax, p in zip(np.atleast_1d(axes), image_paths[:4]):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    rel = os.path.relpath(p, DATA_DIR)
    ax.imshow(img); ax.set_title(f"{rel}\n(true count: {ground_truth[rel]})"); ax.axis('off')
plt.tight_layout(); plt.show()


NameError: name 'glob' is not defined

## 2. Preprocessing

Standard pipeline: resize to a consistent working size (keeps Hough parameters
comparable across images), convert to grayscale, and denoise with a median blur
(median blur preserves circular edges better than Gaussian blur here, and is
robust to the salt-and-pepper texture of coin engravings).


In [ ]:
def preprocess(img, target_max_dim=1000):
    """Resize + grayscale + denoise. Returns (resized_bgr, gray_blurred)."""
    h, w = img.shape[:2]
    scale = target_max_dim / max(h, w)
    img_r = cv2.resize(img, (int(w * scale), int(h * scale)))
    gray = cv2.cvtColor(img_r, cv2.COLOR_BGR2GRAY)
    blurred = cv2.medianBlur(gray, 7)
    return img_r, blurred

# demo
demo_img = cv2.imread(image_paths[0])
demo_resized, demo_gray = preprocess(demo_img)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv2.cvtColor(demo_resized, cv2.COLOR_BGR2RGB)); axes[0].set_title("Resized"); axes[0].axis('off')
axes[1].imshow(demo_gray, cmap='gray'); axes[1].set_title("Grayscale + median blur"); axes[1].axis('off')
plt.tight_layout(); plt.show()


## 3. Classical CV detector — feature extraction + Hough Circle Transform

**Why Hough Circles and not a simple brightness threshold:** coins in this dataset
range from near-black to silver to gold, and some are nearly the same brightness
as the background, so a single global/Otsu threshold fails to separate several
coins from the background (tested and confirmed on this dataset). Coins are
reliably circular though, so working from **edges** (Hough Circle Transform)
rather than **brightness** is far more robust here.

We add non-maximum suppression (NMS) on the returned circles as a feature-based
cleanup step: Hough sometimes fires two overlapping circle hypotheses on the
same coin (especially the engraved/textured ones), so we merge any two detected
circles whose centres are closer than 0.65× the sum of their radii.


In [ ]:
def nms_circles(circles, overlap_thresh=0.65):
    """Greedy NMS: circles are already ordered by Hough accumulator strength.
    Drop any circle whose centre is too close to an already-kept circle."""
    keep = []
    for (x, y, r) in circles:
        is_dup = False
        for (kx, ky, kr) in keep:
            d = np.hypot(x - kx, y - ky)
            if d < overlap_thresh * (r + kr):
                is_dup = True
                break
        if not is_dup:
            keep.append((x, y, r))
    return keep


def detect_coins_classical(img, min_radius=15, max_radius=60, return_vis=False):
    """Detect coins with Hough Circle Transform on a preprocessed image.
    Returns list of (x, y, r) in the coordinate system of the resized image,
    and (optionally) an annotated visualisation image.
    """
    img_r, gray = preprocess(img)

    circles = cv2.HoughCircles(
        gray, cv2.HOUGH_GRADIENT, dp=1.2, minDist=45,
        param1=60, param2=32, minRadius=min_radius, maxRadius=max_radius
    )

    detections = []
    if circles is not None:
        raw = np.round(circles[0]).astype(int)
        detections = nms_circles(raw)

    if not return_vis:
        return detections, img_r

    vis = img_r.copy()
    for i, (x, y, r) in enumerate(detections, start=1):
        cv2.circle(vis, (x, y), r, (0, 255, 0), 3)
        cv2.putText(vis, str(i), (x - 10, y + 8), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
    return detections, vis


# demo on a handful of images
sample = image_paths[:6]
fig, axes = plt.subplots(1, len(sample), figsize=(4 * len(sample), 5))
for ax, p in zip(np.atleast_1d(axes), sample):
    img = cv2.imread(p)
    dets, vis = detect_coins_classical(img, return_vis=True)
    rel = os.path.relpath(p, DATA_DIR)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{rel}\ndetected = {len(dets)} (true = {ground_truth[rel]})")
    ax.axis('off')
plt.tight_layout(); plt.show()


## 4. Baseline evaluation (classical CV vs. true count)

Ground truth is already loaded for all 300 images (Section 1), so we can
evaluate the classical detector on the whole dataset rather than a hand-counted
sample.


In [ ]:
def evaluate_counts(pred_counts: dict, gt: dict):
    rows, errors = [], []
    for rel_path, gt_count in gt.items():
        pred = pred_counts.get(rel_path)
        if pred is None:
            continue
        err = pred - gt_count
        errors.append(abs(err))
        rows.append((rel_path, gt_count, pred, err))
    mae = np.mean(errors) if errors else float('nan')
    exact_acc = np.mean([e == 0 for e in errors]) if errors else float('nan')
    return rows, mae, exact_acc

classical_counts = {}
for p in image_paths:
    dets, _ = detect_coins_classical(cv2.imread(p))
    classical_counts[os.path.relpath(p, DATA_DIR)] = len(dets)

rows, mae, acc = evaluate_counts(classical_counts, ground_truth)
print(f"{'file':20s} {'GT':>4s} {'pred':>5s} {'err':>5s}")
for rel_path, gtc, pred, err in rows[:20]:
    print(f"{rel_path:20s} {gtc:4d} {pred:5d} {err:5d}")
print(f"... ({len(rows)} images total)")
print(f"\nClassical CV baseline — MAE: {mae:.2f}, Exact-match accuracy: {acc*100:.1f}%")

# MAE broken down by true count - useful for spotting where the detector struggles
# (e.g. it may hold up fine at low counts but degrade as coins start overlapping more)
by_count = {}
for rel_path, gtc, pred, err in rows:
    by_count.setdefault(gtc, []).append(abs(err))
print("\nMAE by true coin count:")
for gtc in sorted(by_count):
    print(f"  count={gtc:2d}: MAE={np.mean(by_count[gtc]):.2f}  (n={len(by_count[gtc])})")


## 5. Auto-labelling for the deep-learning model

The dataset has whole-image count labels (from the folder structure) but no
bounding boxes, so we **auto-generate YOLO-format pseudo-labels from the
classical detector's output** (Section 3) — a standard semi-supervised
technique.

Because we now have the *true* count for every image, the filtering is
principled rather than a guess: an image's pseudo-label (its set of detected
circles) is only trusted for training when **the classical detector's count
matches the folder's true count**. If Hough detected 6 circles on an image
folder-labelled as 7, something went wrong (touching coins merged, a false
positive, blur, etc.) and that pseudo-label is unreliable — so it's excluded
from training and kept only for evaluation (Sections 4 and 9 already cover
that on the full dataset).


In [ ]:
YOLO_DATASET_DIR = "./coin_yolo_dataset"
IMAGES_DIR = os.path.join(YOLO_DATASET_DIR, "images")
LABELS_DIR = os.path.join(YOLO_DATASET_DIR, "labels")
for split in ["train", "val"]:
    os.makedirs(os.path.join(IMAGES_DIR, split), exist_ok=True)
    os.makedirs(os.path.join(LABELS_DIR, split), exist_ok=True)


def circles_to_yolo_lines(circles, img_w, img_h, box_scale=1.15):
    """Convert (x, y, r) circles to YOLO 'class cx cy w h' lines (normalised 0-1).
    box_scale slightly enlarges the box beyond the circle to fully cover the coin rim."""
    lines = []
    for (x, y, r) in circles:
        r2 = r * box_scale
        cx, cy = x / img_w, y / img_h
        bw, bh = (2 * r2) / img_w, (2 * r2) / img_h
        cx, cy = min(max(cx, 0), 1), min(max(cy, 0), 1)
        bw, bh = min(bw, 1), min(bh, 1)
        lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
    return lines


pseudo_labelled = []  # (rel_path, resized_bgr, circles) - only trustworthy pseudo-labels
mismatches = []       # (rel_path, true_count, classical_count) - excluded from training
for p, true_count in image_records:
    rel_path = os.path.relpath(p, DATA_DIR)
    img = cv2.imread(p)
    circles, resized = detect_coins_classical(img)
    if len(circles) != true_count:
        mismatches.append((rel_path, true_count, len(circles)))
        continue
    pseudo_labelled.append((rel_path, resized, circles))

print(f"Trustworthy pseudo-labels: {len(pseudo_labelled)} / {len(image_records)} images "
      f"({len(mismatches)} excluded where classical count != true count)")
print("\nA few excluded examples (rel_path, true_count, classical_count):")
for row in mismatches[:10]:
    print(" ", row)

# train/val split (80/20) - only over the trustworthy pseudo-labels
random.shuffle(pseudo_labelled)
n_val = max(1, int(0.2 * len(pseudo_labelled)))
val_set = pseudo_labelled[:n_val]
train_set = pseudo_labelled[n_val:]
print(f"\ntrain: {len(train_set)}  val: {len(val_set)}")


## 6. Data augmentation

10 images per coin-count class (30 classes) is still small for a deep model to
generalise from, so we expand the *training* split with geometry-aware
augmentations (coins are circles, so rotating/flipping the image just moves
the circle centre — the radius and label format stay correct).

**This dataset's labels are exact coin counts, so an augmentation must not
change how many coins are visible.** A coin rotated near the edge of frame
could get pushed out of view, silently making the label wrong. Two guards
against that:
- Rotation is kept to small angles (±15°) rather than large/90° rotations,
  which for these rectangular (4:3) photos would crop much more of the frame.
- After every augmentation we check that the number of *surviving* circles
  still equals the original count — if an augmentation would have changed the
  visible count, that sample is discarded rather than kept with a wrong label.


In [ ]:
def augment_sample(img, circles, angle=0, flip=None, brightness=1.0, contrast=1.0):
    h, w = img.shape[:2]
    out = img.copy()
    new_circles = list(circles)

    if flip == 'h':
        out = cv2.flip(out, 1)
        new_circles = [(w - x, y, r) for (x, y, r) in new_circles]
    elif flip == 'v':
        out = cv2.flip(out, 0)
        new_circles = [(x, h - y, r) for (x, y, r) in new_circles]

    if angle != 0:
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        out = cv2.warpAffine(out, M, (w, h), borderMode=cv2.BORDER_REPLICATE)
        rotated = []
        for (x, y, r) in new_circles:
            vec = M @ np.array([x, y, 1.0])
            nx, ny = vec[0], vec[1]
            # keep only circles that stay fully inside the frame after rotation
            if r <= nx <= w - r and r <= ny <= h - r:
                rotated.append((nx, ny, r))
        new_circles = rotated

    # brightness/contrast jitter - no geometry change
    out = cv2.convertScaleAbs(out, alpha=contrast, beta=(brightness - 1.0) * 60)

    return out, new_circles


AUGS_PER_IMAGE = 6

def build_split(samples, split_name, augment: bool):
    img_out_dir = os.path.join(IMAGES_DIR, split_name)
    lbl_out_dir = os.path.join(LABELS_DIR, split_name)
    count, discarded = 0, 0
    for rel_path, img, circles in samples:
        h, w = img.shape[:2]
        # rel_path looks like "7/IMG003.jpg" - flatten to a safe filename stem
        stem = rel_path.replace("/", "_").replace("\\", "_")
        stem = os.path.splitext(stem)[0]

        # original (always included)
        cv2.imwrite(os.path.join(img_out_dir, f"{stem}_orig.jpg"), img)
        with open(os.path.join(lbl_out_dir, f"{stem}_orig.txt"), "w") as f:
            f.write("\n".join(circles_to_yolo_lines(circles, w, h)))
        count += 1

        if augment:
            for i in range(AUGS_PER_IMAGE):
                angle = random.uniform(-15, 15)
                flip = random.choice([None, 'h', 'v'])
                brightness = random.uniform(0.85, 1.15)
                contrast = random.uniform(0.9, 1.1)
                aug_img, aug_circles = augment_sample(img, circles, angle, flip, brightness, contrast)

                # discard if this augmentation changed the visible coin count -
                # keeping it would silently attach a wrong label to the image
                if len(aug_circles) != len(circles):
                    discarded += 1
                    continue

                out_name = f"{stem}_aug{i}"
                cv2.imwrite(os.path.join(img_out_dir, f"{out_name}.jpg"), aug_img)
                with open(os.path.join(lbl_out_dir, f"{out_name}.txt"), "w") as f:
                    f.write("\n".join(circles_to_yolo_lines(aug_circles, w, h)))
                count += 1
    return count, discarded

n_train, disc_train = build_split(train_set, "train", augment=True)
n_val, _ = build_split(val_set, "val", augment=False)  # keep val clean/un-augmented
print(f"Wrote {n_train} training images ({disc_train} augmentations discarded "
      f"for changing the visible coin count), {n_val} validation images")


## 7. GPU-accelerated training — YOLOv8n (Ultralytics / PyTorch, CUDA)

`YOLOv8n` is the smallest YOLOv8 variant — fast to fine-tune on a small dataset
and on a single local GPU. We start from COCO-pretrained weights (transfer
learning) rather than training from scratch, since our pseudo-labelled set is
small. The first run will download `yolov8n.pt` automatically (needs internet
once).

If you're on CPU only, drop `epochs` to ~15-20 and `imgsz` to 416 so it finishes
in reasonable time.


In [ ]:
data_yaml = f"""
path: {os.path.abspath(YOLO_DATASET_DIR)}
train: images/train
val: images/val
nc: 1
names: ['coin']
"""
with open(os.path.join(YOLO_DATASET_DIR, "data.yaml"), "w") as f:
    f.write(data_yaml)

from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else 'cpu'
model = YOLO("yolov8n.pt")  # COCO-pretrained weights, fine-tuned below

results = model.train(
    data=os.path.join(YOLO_DATASET_DIR, "data.yaml"),
    epochs=60,
    imgsz=640,
    batch=8,
    device=device,
    patience=15,
    project="coin_counter_runs",
    name="yolov8n_coins",
)


## 8. Run the trained detector and count coins


In [ ]:
best_weights = os.path.join(results.save_dir, "weights", "best.pt")
trained_model = YOLO(best_weights)

def detect_coins_yolo(img_path, conf=0.35):
    res = trained_model.predict(img_path, conf=conf, verbose=False)[0]
    boxes = res.boxes.xyxy.cpu().numpy() if res.boxes is not None else []
    return len(boxes), res

yolo_counts = {}
sample = image_paths[:6]
fig, axes = plt.subplots(1, len(sample), figsize=(4 * len(sample), 5))
for ax, p in zip(np.atleast_1d(axes), sample):
    count, res = detect_coins_yolo(p)
    rel_path = os.path.relpath(p, DATA_DIR)
    yolo_counts[rel_path] = count
    vis = res.plot()
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{rel_path}\nYOLO count = {count} (true = {ground_truth[rel_path]})")
    ax.axis('off')
plt.tight_layout(); plt.show()

# run on the full dataset for the evaluation table below
for p in image_paths:
    rel_path = os.path.relpath(p, DATA_DIR)
    if rel_path not in yolo_counts:
        count, _ = detect_coins_yolo(p)
        yolo_counts[rel_path] = count


## 9. Comparative evaluation: classical CV vs. YOLOv8 vs. your ground truth


In [ ]:
print(f"{'file':20s} {'GT':>4s} {'classical':>10s} {'YOLO':>6s}")
c_errors, y_errors = [], []
for rel_path, gt_count in ground_truth.items():
    c_pred = classical_counts.get(rel_path)
    y_pred = yolo_counts.get(rel_path)
    if len(c_errors) < 30:  # full 300-row dump is a lot - print a sample, use all rows for the stats below
        print(f"{rel_path:20s} {gt_count:4d} {c_pred!s:>10s} {y_pred!s:>6s}")
    if c_pred is not None:
        c_errors.append(abs(c_pred - gt_count))
    if y_pred is not None:
        y_errors.append(abs(y_pred - gt_count))

if c_errors:
    print(f"\nClassical CV  — MAE: {np.mean(c_errors):.2f}, "
          f"Exact-match accuracy: {np.mean([e==0 for e in c_errors])*100:.1f}%")
if y_errors:
    print(f"YOLOv8        — MAE: {np.mean(y_errors):.2f}, "
          f"Exact-match accuracy: {np.mean([e==0 for e in y_errors])*100:.1f}%")

# Chart 1: per-image bars for one image per true-count class (30 bars, readable)
one_per_class = {}
for rel_path, gt_count in ground_truth.items():
    one_per_class.setdefault(gt_count, rel_path)
labels = [one_per_class[k] for k in sorted(one_per_class)]
gt_vals = [ground_truth[k] for k in labels]
c_vals = [classical_counts.get(k, 0) for k in labels]
y_vals = [yolo_counts.get(k, 0) for k in labels]
x = np.arange(len(labels))
w = 0.25
plt.figure(figsize=(max(10, len(labels) * 0.6), 5))
plt.bar(x - w, gt_vals, w, label='Ground truth')
plt.bar(x, c_vals, w, label='Classical CV')
plt.bar(x + w, y_vals, w, label='YOLOv8')
plt.xticks(x, [str(ground_truth[l]) for l in labels])
plt.xlabel('True coin count (one example image per count shown)')
plt.ylabel('Predicted coin count'); plt.legend(); plt.tight_layout()
plt.savefig('./evaluation_chart_by_count.png', dpi=150)
plt.show()

# Chart 2: MAE as a function of true coin count, over the FULL dataset - shows
# whether accuracy degrades as more coins (and more overlap) appear in frame
def mae_by_count(pred_counts):
    buckets = {}
    for rel_path, gt_count in ground_truth.items():
        pred = pred_counts.get(rel_path)
        if pred is not None:
            buckets.setdefault(gt_count, []).append(abs(pred - gt_count))
    counts = sorted(buckets)
    return counts, [np.mean(buckets[c]) for c in counts]

c_x, c_mae = mae_by_count(classical_counts)
y_x, y_mae = mae_by_count(yolo_counts)
plt.figure(figsize=(10, 5))
plt.plot(c_x, c_mae, marker='o', label='Classical CV')
plt.plot(y_x, y_mae, marker='o', label='YOLOv8')
plt.xlabel('True coin count'); plt.ylabel('Mean absolute error')
plt.legend(); plt.tight_layout()
plt.savefig('./mae_by_count.png', dpi=150)
plt.show()
